<a href="https://colab.research.google.com/github/Serge3leo/temp-cola/blob/main/valq/260325/ecosw_sin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from math import *
import locale
import os
import sys
import time

import astropy.time as at
import numpy as np
import scipy
import matplotlib.pyplot as plt
import tqdm

%matplotlib osx
# %config InlineBackend.figure_format = "retina"

In [2]:
locale.setlocale(locale.LC_ALL, 'ru_RU.UTF-8')
time.strftime("%x %X", time.localtime())

'03.04.2026 05:41:22'

In [3]:
# Хм, а скопировать что-ли? Пока, ячейка с `lss_error` на colab 
# работать не будет, надо будет пропускать руками
hdr_add_modules = os.environ['HOME'] + '/soft/valq/modules'

if hdr_add_modules not in sys.path:
    assert 'hdr_save_sys_path' not in globals(), \
            "hdr_add_modules нельзя удалять из sys.path"
    hdr_save_sys_path = sys.path.copy()
    sys.path.insert(1, hdr_add_modules)
elif hdr_add_modules in sys.path:
    print('hdr_add_modules уже есть в sys.path')
else:
    assert False, "Ошибка, однако"

In [4]:
import lss_error
# np.int = np.int64  # dfogn workaround
# lss_error.do_test()  # verbose=True)

# Входные данные

In [5]:
a_ecw = 0
a_ecw_sigma = 1
a_jdh = 2
a_jdh_sigma = 3

```
 "ecosw""sigmaecosw"   "jdh""sigmajdh"
...
система уравнений : e*cos(w0+(jdh-jdh0)*dw)  c ошибками ecos и jdh
```

In [6]:
"""
27 марта 2026 г., в 11:51, Valentina Kozyreva <valiakozyreva@gmail.com> написал(а):

привет. надыбала из литературы ( сгруппировав близкие по jdh с разбросом) еще несколько точек. Посмотри, это в ДЕЙСТВИТЕЛЬНЫХ решается
<ecosw>
"""
data_set_id = "ecosw 260327"
ecosw_in = np.loadtxt("ecosw.txt", skiprows=1, usecols=(0,1,2,3),
                      max_rows=7)
ecosw = ecosw_in  # [(0,3,4,5,6),:] # Если убрать пару точек, то совсем неубедительно
with np.printoptions(precision=6, suppress=True):
    for el in ecosw:
        print(el, at.Time(el[a_jdh], format='mjd').iso)
jdh0 = ecosw[0, a_jdh]
print(f"{a_ecw, a_jdh, jdh0=}")

[   -0.00061     0.00487 52450.         24.     ] 2002-06-25 00:00:00.000
[    0.00192     0.00308 53900.        470.     ] 2006-06-14 00:00:00.000
[    0.00723     0.00498 56522.        189.     ] 2013-08-18 00:00:00.000
[    0.00566     0.0044  57920.         10.     ] 2017-06-16 00:00:00.000
[    0.00492     0.00092 59011.         15.     ] 2020-06-11 00:00:00.000
[    0.00435     0.00097 59753.         10.     ] 2022-06-23 00:00:00.000
[    0.00479     0.00123 60479.         15.     ] 2024-06-18 00:00:00.000
a_ecw, a_jdh, jdh0=(0, 2, 52450.0)


## Учёт ошибок входных данных

В данном случае, взвешивание на $1 / \sigma_{e \cos(\omega)}$ даёт
весьма консервативный результат. Типа надёжно, и без всякого
излишнего оптимизма.

In [7]:
# weight = np.ones_like(ecosw[:, a_ecw_sigma])

weight = np.full_like(ecosw[:, a_ecw_sigma],
                      np.sqrt(ecosw.shape[0]/np.sum(ecosw[:, a_ecw_sigma]**2)))
weight = 1./ecosw[:, a_ecw_sigma]

# weight = np.full_like(ecosw[:, a_ecw_sigma],
#                       ecosw.shape[0]/np.sqrt(np.sum(ecosw[:, a_ecw_sigma]**2)))
# weight = np.sqrt(ecosw.shape[0])/ecosw[:, a_ecw_sigma]

weight_txt = "Без весов" if np.all(weight[0] == weight) else "Взвешенное"

# Расчёт невязок и матрицы Якоби

In [8]:
method_id = ('e', 'w0', 'dw')
x_e = 0
x_w0 = 1
x_dw = 2
def ecw_fun_c(x, ecw_=ecosw[:, a_ecw], jdh_=ecosw[:, a_jdh],
              jdh0_=jdh0, weight_=weight):
    return ((x[x_e]*np.cos(x[x_w0] + (jdh_ - jdh0_)*x[x_dw]))
            - ecw_) * weight_

def ecw_jac_c(x, ecw_=ecosw[:, a_ecw], jdh_=ecosw[:, a_jdh],
              jdh0_=jdh0, weight_=weight):
    jac = np.empty((ecosw.shape[0], len(x)))
    djdh = jdh_ - jdh0_
    jac[:,0] = np.cos(x[x_w0] + djdh*x[x_dw]) * weight_
    jac[:,1] = -x[x_e]*np.sin(x[x_w0] + djdh*x[x_dw]) * weight_
    jac[:,2] = jac[:,1]*djdh
    return jac

In [9]:
x_scale = 'jac'
f_scale = np.sqrt(np.sum((ecosw[:, a_ecw]*weight)**2))  # TODO: weight
rcnt = 7
print(f"{x_scale, f_scale, rcnt=}")
def rcnt_solsq_1(fun_, x0_, jac_, bounds_, method_, kwargs_):
    return scipy.optimize.least_squares(fun_, x0_, jac=jac_,
            bounds=bounds_ if method_ != 'lm' else {-np.inf, np.inf},
            method=method_, kwargs=kwargs_,
            x_scale=x_scale, f_scale=f_scale,
            xtol=1e-15, ftol=1e-15, gtol=1e-13)
def rcnt_solsq(fun_, x0_, jac_, bounds_, method_, kwargs_, rcnt_):
    r0 = rcnt_solsq_1(fun_, x0_, jac_, bounds_, method_, kwargs_)
    assert np.all(r0.x >= bounds_[0]), f"{r0, r0.x, bounds_[0]=}"
    assert np.all(r0.x <= bounds_[1]), f"{r0, r0.x, bounds_[1]=}"
    for _ in range(rcnt_):
        xc = np.random.uniform(low=bounds_[0], high=bounds_[1])
        rc = rcnt_solsq_1(fun_, xc, jac_, bounds_, method_, kwargs_)
        if rc.cost < r0.cost:
            assert np.all(rc.x >= bounds_[0]), f"{rc, rc.x, bounds_[0]=}"
            assert np.all(rc.x <= bounds_[1]), f"{rc, rc.x, bounds_[1]=}"
            # print(f"{r0.cost - rc.cost}")
            r0 = rc
    return r0

x_scale, f_scale, rcnt=('jac', 8.248800710363556, 7)


In [10]:
def solsq(x0_, bounds_, use_jac=True, method_='dogbox',
          ecw_=ecosw[:, a_ecw], jdh_=ecosw[:, a_jdh], jdh0_=jdh0,
          weight_=weight, rcnt_=0):
    jac = ecw_jac_c if use_jac else '2-point'  # {‘2-point’, ‘3-point’, ‘cs’, callable}, optional
    return rcnt_solsq(fun_=ecw_fun_c, x0_=x0_, jac_=jac,
            bounds_=bounds_, method_=method_,
            kwargs_=dict(ecw_=ecw_, jdh_=jdh_, jdh0_=jdh0_,
                         weight_=weight_),
            rcnt_=rcnt_)

## Вариант замены переменной
$\omega_0 = \omega_0' - \frac{\pi}{2}$

После замены, графики немного ровнее, но это видно только на большом
масштабе. Т.е. результаты, пока, идентичны. Только расчёт немного
шустрее.

In [11]:
method_id = ('e', "w0'", 'dw')
x_e = 0
x_w0_s = 1
x_dw = 2
def ecw_fun_s(x, ecw_=ecosw[:, a_ecw], jdh_=ecosw[:, a_jdh],
              jdh0_=jdh0, weight_=weight):
    return ((x[x_e]*np.sin(x[x_w0_s] + (jdh_ - jdh0_)*x[x_dw]))
            - ecw_) * weight_

def ecw_jac_s(x, ecw_=ecosw[:, a_ecw], jdh_=ecosw[:, a_jdh],
              jdh0_=jdh0, weight_=weight):
    jac = np.empty((ecosw.shape[0], len(x)))
    djdh = jdh_ - jdh0_
    jac[:,0] = np.sin(x[x_w0_s] + djdh*x[x_dw]) * weight_
    jac[:,1] = x[x_e]*np.cos(x[x_w0_s] + djdh*x[x_dw]) * weight_
    jac[:,2] = jac[:,1]*djdh
    return jac

In [12]:
def solsq(x0_, bounds_, use_jac=True, method_='dogbox',
          ecw_=ecosw[:, a_ecw], jdh_=ecosw[:, a_jdh], jdh0_=jdh0,
          weight_=weight, rcnt_=0):
    jac = ecw_jac_s if use_jac else '2-point'  # {‘2-point’, ‘3-point’, ‘cs’, callable}, optional
    return rcnt_solsq(fun_=ecw_fun_s, x0_=x0_, jac_=jac,
            bounds_=bounds_, method_=method_,
            kwargs_=dict(ecw_=ecw_, jdh_=jdh_, jdh0_=jdh0_,
                         weight_=weight_),
            rcnt_=rcnt_)

In [13]:
if "w0'" == method_id[1]:
    add2cos = np.array([0., -np.pi/2., 0.])
else:
    add2cos = np.zeros(3)

# Начальная точка
Есть вопросы, но пока так.

In [14]:
x0 = np.array((np.average(ecosw[:, a_ecw]), np.pi/2.-0.001, 0.001))
x0

array([4.03714286e-03, 1.56979633e+00, 1.00000000e-03])

In [15]:
# minres_a.x, minres_a
# minres_b.x, minres_b

In [16]:
if "ecosw 260327" == data_set_id:
    x0 = np.array([4.99129963e-03, 4.75366679e+00 - 2*pi, 2.54988863e-04])
    print(x0)

[ 4.99129963e-03 -1.52951852e+00  2.54988863e-04]


In [17]:
if "ecosw 260327" == data_set_id:
    if np.all(weight[0] == weight):
        x0 = np.array([0.006297200962871222, -0.09100045390164574, 0.00032152156628824065])
        x0 = np.array([0.006297200962871222, -0.09100045390164468, 0.00032152156628824065])
        if "w0" == method_id[1]:
            x0[1] += -np.pi/2.
    else:
        x0 = np.array([4.99129963e-03, 0.04127780961530991, 2.54988863e-04])
        x0 = np.array([0.004954477174761145, 0.07688715100396747, 0.017458411894649924])
        x0 = np.array([0.004954477179025791, -6.2062981611627706 + 2*pi, 0.01745841189584378])
        x0 = np.array([0.004954477179025791, 0.07688714601682278, 0.01745841189584378])
        if "w0" == method_id[1]:
            x0[1] += -np.pi/2.
            x0 = np.array([0.0049544771645790645, -1.493909170419644, 0.01745841189261346])
            x0 = np.array([0.00495447715987243, -1.4939091737708958, 0.017458411892398665])
    print(x0)

[0.00495448 0.07688715 0.01745841]


In [18]:
if "V1097 Her 250401" == data_set_id:
    x0 = np.array([ 0.01403113,  2.74652896, -0.03979614])
    x0 = np.array([ 0.01403113, -5.88812162 + 2*pi,  0.03979614])
    print(x0)

In [19]:
z = x0[1] + pi/2
z

1.6476834728117193

In [20]:
x0[1], z - pi/2

(0.07688714601682278, 0.07688714601682278)

# Границы
Первоначально, поиск не уходил с нуля. Пока оставил ненулевые ограничения.

Ограничение $\omega \in [-2 \pi ... 2 \pi]$ , выбрано таковым, что бы 
поиск не упирался в него.

In [21]:
bounds = np.array(((1e-7, -2.*np.pi, -0.1),
                   (0.5, 2.*np.pi, 0.1)))
bounds

array([[ 1.00000000e-07, -6.28318531e+00, -1.00000000e-01],
       [ 5.00000000e-01,  6.28318531e+00,  1.00000000e-01]])

# Поиск и сравнение матриц Якоби

In [22]:
res = solsq(x0, bounds, method_='lm')
res

     message: `xtol` termination condition is satisfied.
     success: True
      status: 3
         fun: [ 2.034e-01 -2.116e-01 -5.701e-01 -1.937e-01  3.163e-02
                3.288e-01 -2.666e-01]
           x: [ 4.954e-03  7.689e-02  1.746e-02]
        cost: 0.3144667639160945
         jac: [[ 1.577e+01  1.014e+00  0.000e+00]
               [ 8.310e+01  1.555e+00  2.255e+03]
               ...
               [ 9.715e+02 -1.709e+00 -1.248e+04]
               [ 7.322e+02 -1.751e+00 -1.406e+04]]
        grad: [-6.014e-11  4.798e-09  1.451e-04]
  optimality: 0.00014507760397464153
 active_mask: [0 0 0]
        nfev: 7
        njev: 1

In [23]:
res = solsq(x0, bounds, method_='lm', rcnt_=rcnt)
res

     message: `xtol` termination condition is satisfied.
     success: True
      status: 3
         fun: [ 2.034e-01 -2.116e-01 -5.701e-01 -1.937e-01  3.163e-02
                3.288e-01 -2.666e-01]
           x: [ 4.954e-03  7.689e-02  1.746e-02]
        cost: 0.3144667639160945
         jac: [[ 1.577e+01  1.014e+00  0.000e+00]
               [ 8.310e+01  1.555e+00  2.255e+03]
               ...
               [ 9.715e+02 -1.709e+00 -1.248e+04]
               [ 7.322e+02 -1.751e+00 -1.406e+04]]
        grad: [-6.014e-11  4.798e-09  1.451e-04]
  optimality: 0.00014507760397464153
 active_mask: [0 0 0]
        nfev: 7
        njev: 1

In [24]:
resn = solsq(x0, bounds, use_jac=False, method_='lm')
resn

     message: `xtol` termination condition is satisfied.
     success: True
      status: 3
         fun: [ 2.034e-01 -2.116e-01 -5.701e-01 -1.937e-01  3.163e-02
                3.288e-01 -2.666e-01]
           x: [ 4.954e-03  7.689e-02  1.746e-02]
        cost: 0.3144667639160905
         jac: [[ 1.577e+01  1.014e+00  0.000e+00]
               [ 8.310e+01  1.555e+00  2.255e+03]
               ...
               [ 9.715e+02 -1.709e+00 -1.248e+04]
               [ 7.322e+02 -1.751e+00 -1.405e+04]]
        grad: [ 3.322e-06 -8.851e-09 -2.018e-01]
  optimality: 0.20183716710266708
 active_mask: [0 0 0]
        nfev: 20
        njev: None

In [25]:
resn = solsq(x0, bounds, use_jac=False, method_='lm', rcnt_=rcnt)
resn

     message: `xtol` termination condition is satisfied.
     success: True
      status: 3
         fun: [ 2.034e-01 -2.116e-01 -5.701e-01 -1.937e-01  3.163e-02
                3.288e-01 -2.666e-01]
           x: [ 4.954e-03  7.689e-02  1.746e-02]
        cost: 0.3144667639160905
         jac: [[ 1.577e+01  1.014e+00  0.000e+00]
               [ 8.310e+01  1.555e+00  2.255e+03]
               ...
               [ 9.715e+02 -1.709e+00 -1.248e+04]
               [ 7.322e+02 -1.751e+00 -1.405e+04]]
        grad: [ 3.322e-06 -8.851e-09 -2.018e-01]
  optimality: 0.20183716710266708
 active_mask: [0 0 0]
        nfev: 20
        njev: None

In [26]:
(res.x - resn.x)*2./np.abs(res.x + resn.x)

array([ 1.26240273e-12,  1.97479950e-08, -1.47613953e-12])

In [27]:
(res.jac - resn.jac)*2./np.abs(res.jac + resn.jac)

/var/folders/wy/z0gbkfgs7mv24ryqkdm90rm40009rh/T/ipykernel_71561/2699449834.py:1: RuntimeWarning: invalid value encountered in divide
  (res.jac - resn.jac)*2./np.abs(res.jac + resn.jac)


array([[ 1.97090457e-08, -1.08899267e-10,             nan],
       [ 5.59326867e-09, -4.05606350e-10,  1.28489057e-05],
       [-7.38922722e-10, -3.25876377e-09, -1.01339362e-04],
       [ 3.43148287e-10, -4.97263329e-09,  1.82860390e-04],
       [ 6.29525536e-11, -2.83652492e-08,  2.63061566e-04],
       [-4.72296840e-10, -4.31012168e-09, -3.25970379e-04],
       [-6.32874888e-10, -3.31505884e-09, -3.94005573e-04]])

In [28]:
if "w0" != method_id[1]:
    print((resn.jac - ecw_jac_c(resn.x - [0., pi/2, 0.]))
          /ecw_jac_c(resn.x - [0., pi/2, 0.]))

[[ 1.93714385e-14 -6.81189574e-12             nan]
 [-1.35089561e-14  1.47246861e-11 -1.28492140e-05]
 [ 1.38951779e-14 -5.53843278e-10 -1.01331523e-04]
 [ 2.22959722e-14 -5.54492932e-10 -1.82849199e-04]
 [-2.57575722e-14 -5.63920617e-10 -2.63055891e-04]
 [ 2.37552084e-14 -5.62461862e-10 -3.25913513e-04]
 [ 1.04027626e-14 -5.96119885e-10 -3.93925250e-04]]


/var/folders/wy/z0gbkfgs7mv24ryqkdm90rm40009rh/T/ipykernel_71561/2486576611.py:2: RuntimeWarning: invalid value encountered in divide
  print((resn.jac - ecw_jac_c(resn.x - [0., pi/2, 0.]))


Не похоже, что бы была явная лажа в вычислении матрицы Якоби. Более-менее.

In [29]:
np.cos(resn.x[x_w0] + add2cos[x_w0] + (ecosw[:, a_jdh] - jdh0)*resn.x[x_dw])

array([0.07681141, 0.25596243, 0.88620599, 0.97033693, 0.99891422,
       0.94236493, 0.90061964])

In [30]:
np.cos(res.x[x_w0] + add2cos[x_w0] + (ecosw[:, a_jdh] - jdh0)*res.x[x_dw])

array([0.07681141, 0.25596243, 0.88620599, 0.97033693, 0.99891422,
       0.94236493, 0.90061964])

Замечание до замены:
> Три числа около 1 намекают, что стоит попробовать сделать
замену: $\omega_0 = \omega_0' - \frac{\pi}{2}$

Смысл замены так пока и не подтвердился, пока не ясен.

# Типа невязки
Пока выглядит всё более менее.  Однако, решения с весами и без весов
заметно различаются.

Взвешенные невязки, в $\sigma$ каждой точки.

In [31]:
(res.fun / weight) / ecosw[:, a_ecw_sigma]

array([ 0.20340049, -0.21163636, -0.57014311, -0.19374723,  0.03162793,
        0.3287892 , -0.26658582])

In [32]:
ecosw[:, a_jdh_sigma]

array([ 24., 470., 189.,  10.,  15.,  10.,  15.])

In [33]:
np.sqrt(np.sum(res.fun**2)), np.sqrt(2.*res.cost)

(0.7930532944463373, 0.7930532944463373)

Невязки без весов, так сказать, а ля натурель.

In [34]:
res.fun/weight

array([ 9.90560396e-04, -6.51839994e-04, -2.83931267e-03, -8.52487824e-04,
        2.90976967e-05,  3.18925528e-04, -3.27900559e-04])

ecosw 260327:
- При оценке без весов, невязка меньше оценки ошибки в 4,5 раза;
- При взвешенной оценке, невязка меньше оценки ошибки в 2,8 раза
  ($\approx \sqrt7$), что бы это значило?

In [35]:
np.linalg.norm(ecosw[:, a_ecw_sigma])/np.linalg.norm(res.fun/weight)

2.7843676326535762

# Оценка точности решения МНК
По матрице Якоби (статистическая). Статистика, конечно, никакая, да и
минимум неглубокий, в общем, сплошной розовый оптимизм, но... 

In [36]:
err, cov = lss_error.scipyERR(res)
with np.printoptions(precision=5):
    print(err)
    print(cov)
    print(f"{data_set_id}, {method_id[1]}, {weight_txt}")
    for i in range(3):
        print(f"{method_id[i]:4} = {res.x[i]: .5f} ± {err[i]:.5f} (1𝜎, 68%)")
        if "w0'" == method_id[i]:
            print(f"{'w0':4} = {res.x[i] + add2cos[i]: .5f} ± {err[i]:.5f} (1𝜎, 68%)")

[3.16255e-04 2.45919e-01 4.57275e-05]
[[ 1.00017e-07 -1.14083e-05  7.16447e-09]
 [-1.14083e-05  6.04763e-02 -9.06366e-06]
 [ 7.16447e-09 -9.06366e-06  2.09100e-09]]
ecosw 260327, w0', Взвешенное
e    =  0.00495 ± 0.00032 (1𝜎, 68%)
w0'  =  0.07689 ± 0.24592 (1𝜎, 68%)
w0   = -1.49391 ± 0.24592 (1𝜎, 68%)
dw   =  0.01746 ± 0.00005 (1𝜎, 68%)


# Зависимость суммы квадратов невязок от переменных

Разбиваем область значений каждой переменной на интервалы, и решаем
задачу МНК с ограничением в этом интервале.

Т.е. пытаемся таким образом визуализировать доверительные интервалы.

In [37]:
def bound_plot(res_, bounds_, log=True):
    start = time.perf_counter()
    minres = res_
    fig, ax = plt.subplots(3, 3)
    for i in range(3):
        resj = res_
        boundi = bounds_[:, i].copy()
        lowi = np.linspace(boundi[0], boundi[1], num=128)
        costi = np.zeros((len(lowi) - 1,))
        res_xi = np.zeros((len(lowi) - 1, 3))
        for j in range(len(lowi) - 1):
            xj = res.x.copy()  # TODO ???
            boundj = bounds_.copy()
            for k in range(3):
                assert boundj[0, k] < xj[k] and xj[k] < boundj[1, k], \
                    f"{k, boundj[0, k], xj[k], boundj[1, k]=}"
            boundj[0, i] = lowi[j]
            boundj[1, i] = lowi[j + 1]
            xj[i] = np.average(boundj[:, i])
            for k in range(3):
                assert boundj[0, k] < xj[k] and xj[k] < boundj[1, k], \
                    f"{k, boundj[0, k], xj[k], boundj[1, k]=}"
            resj = solsq(xj, boundj, rcnt_=rcnt)
            costi[j] = resj.cost
            res_xi[j] = resj.x
            if resj.cost < minres.cost:
                minres = resj
        ax[0][i].plot(res_xi[:,i], np.sqrt(2.*costi),
                      label=f"$\\sigma$ of {method_id[i]}")
        if log:
            ax[0][i].set_yscale('log')
        ax[0][i].legend()
        for j in range(3):
            if j != i:
                axji = ax[j + 1 if j < i else j][i]
                axji.plot(res_xi[:,i], res_xi[:,j],
                              label=f"{method_id[j]} of {method_id[i]}")
                axji.legend()
                # axji.set_ylabel(f"{method.id[j]}")
        axji.set_xlabel(f"{method_id[i]}")

    stop = time.perf_counter()
    fig.suptitle(f"{data_set_id}, {method_id[1]}, {weight_txt}"
                 f"\n{time.strftime("%x %X", time.localtime())}"
                 f" ({stop - start:.1f})")
    return fig, ax, minres

## По всей области значений
Верхний ряд это $\sqrt{\sum{{O-C}^2}}$, в зависимости от
изменения соответствующего параметра.

На графиках нижних двух рядов отображаются найденные значения двух
других параметров (свободных параметров).

1. Эксцентриситет, невязка значительно растёт до $e = 0.055..0.059$,
   однако после, устанавливается на уровне $1.6 \times \sigma$. Т.е.
   решение уравнения, само по себе, не даёт надёжных доказательств,
   что он небольшой.
3. $\omega_0$ определяется хреново;
4. `dw` зависит от остальных параметров.

In [38]:
fig, ax, minres_a = bound_plot(res, bounds, log=True)
if minres_a.cost < res.cost:
    print(minres_a)
    print(list(minres_a.x))

In [39]:
if "w0'" == method_id[1]:
    ecw_fun = ecw_fun_s
else:
    ecw_fun = ecw_fun_c
xm0 = minres_a.x.copy()
if xm0[1] > np.pi:
    xm0[1] -= 2*np.pi
elif xm0[1] < -np.pi:
    xm0[1] += 2*np.pi
xm = xm0.copy()
fm = np.sum(ecw_fun(xm)**2)*0.5
print(list(xm), fm, fm <= np.sum(ecw_fun(minres_a.x)**2)*0.5)
for d in np.inf, -np.inf:
    for s in range(1, 1000):
        xt = xm0.copy()
        xt[1] = nextafter(xm0[1], d, steps=s)
        ft = np.sum(ecw_fun(xt)**2)*0.5
        if ft < fm:
            xm = xt
            fm = ft
print(list(xm), fm, fm <= np.sum(ecw_fun(minres_a.x)**2)*0.5)
print(fm - np.sum(ecw_fun(res.x)**2)*0.5)

[0.004954477179025791, 0.07688714601682278, 0.01745841189584378] 0.3144667639160945 True
[0.004954477179025791, 0.07688714601682278, 0.01745841189584378] 0.3144667639160945 True
0.0


In [40]:
xm0 = res.x.copy()
if xm0[1] > np.pi:
    xm0[1] -= 2*np.pi
    print(xm0[1])
elif xm0[1] < -np.pi:
    xm0[1] += 2*np.pi
    print(xm0[1])
xm = xm0.copy()
fm = np.sum(ecw_fun(xm)**2)*0.5
print(list(xm), fm, fm <= np.sum(ecw_fun(res.x)**2)*0.5)
for d in np.inf, -np.inf:
    for s in range(1, 1000):
        xt = xm0.copy()
        xt[1] = nextafter(xm0[1], d, steps=s)
        ft = np.sum(ecw_fun(xt)**2)*0.5
        if ft < fm:
            xm = xt
            fm = ft
print(list(xm), fm, fm <= np.sum(ecw_fun(res.x)**2)*0.5)
print(fm - np.sum(ecw_fun(res.x)**2)*0.5)

[0.004954477179025791, 0.07688714601682278, 0.01745841189584378] 0.3144667639160945 True
[0.004954477179025791, 0.07688714601682278, 0.01745841189584378] 0.3144667639160945 True
0.0


In [41]:
bounds_05_10 = bounds.copy()
bounds_05_10[:,0] = [0.5, 1.0]
x0_05_10 = x0.copy()
x0_05_10[0] = 0.75
res_05_10 = solsq(x0_05_10, bounds_05_10)
res_05_10, np.sqrt(2.*res_05_10.cost/len(res_05_10.fun))

(     message: `ftol` termination condition is satisfied.
      success: True
       status: 2
          fun: [-1.756e+01  3.881e+00 -8.970e+01  1.137e+00  5.529e+01
                -4.571e+01 -2.605e+00]
            x: [ 5.000e-01  3.315e+00  1.719e-02]
         cost: 6761.914227803548
          jac: [[-3.537e+01 -1.011e+02 -0.000e+00]
                [ 9.009e+00 -1.623e+02 -2.353e+05]
                ...
                [-8.246e+01 -5.138e+02 -3.752e+06]
                [ 2.578e+00 -4.065e+02 -3.264e+06]]
         grad: [ 2.696e+04  6.119e-04  2.501e+00]
   optimality: 2.5005943290889263
  active_mask: [-1  0  0]
         nfev: 28
         njev: 19,
 43.954243181496786)

## В районе минимума
1. Эксцентриситет, если есть основания полагать, что он небольшой, то
   на уровне $3 \sigma$ он лежит в интервале `0.003...0.009`;
2. $\omega_0$ видны два минимума: `-1.530` и `+1.311`, их глубины
   $0.8 \times \sigma$ и $0.9 \times \sigma$, разделены горбом
   $1.6 \times \sigma$;

In [42]:
res_bound = np.array((
        (res.x[0] - 0.7*abs(res.x[0]),
         -pi,
         res.x[2] - 3.0*abs(res.x[2])
        ),
        (res.x[0] + 2.0*abs(res.x[0]),
         pi,
         res.x[2] + 2.0*abs(res.x[2])
        )
    ))
fig, ax, minres_b = bound_plot(res, res_bound, log=False)
if minres_b.cost < res.cost and minres_b.cost < minres_a.cost:
    print(minres_b)
    print(list(minres_b.x))

     message: `xtol` termination condition is satisfied.
     success: True
      status: 3
         fun: [ 2.034e-01 -2.116e-01 -5.701e-01 -1.937e-01  3.163e-02
                3.288e-01 -2.666e-01]
           x: [ 4.954e-03  3.065e+00 -1.746e-02]
        cost: 0.3144667639160925
         jac: [[ 1.577e+01 -1.014e+00 -0.000e+00]
               [ 8.310e+01 -1.555e+00 -2.255e+03]
               ...
               [ 9.715e+02  1.709e+00  1.248e+04]
               [ 7.322e+02  1.751e+00  1.406e+04]]
        grad: [-1.219e-09 -1.395e-07 -9.338e-04]
  optimality: 0.0009337610026705079
 active_mask: [0 0 0]
        nfev: 32
        njev: 21
[0.00495447720728342, 3.0647055024083594, -0.01745841189899594]


In [43]:
print(res)
print(list(res.x))

     message: `xtol` termination condition is satisfied.
     success: True
      status: 3
         fun: [ 2.034e-01 -2.116e-01 -5.701e-01 -1.937e-01  3.163e-02
                3.288e-01 -2.666e-01]
           x: [ 4.954e-03  7.689e-02  1.746e-02]
        cost: 0.3144667639160945
         jac: [[ 1.577e+01  1.014e+00  0.000e+00]
               [ 8.310e+01  1.555e+00  2.255e+03]
               ...
               [ 9.715e+02 -1.709e+00 -1.248e+04]
               [ 7.322e+02 -1.751e+00 -1.406e+04]]
        grad: [-6.014e-11  4.798e-09  1.451e-04]
  optimality: 0.00014507760397464153
 active_mask: [0 0 0]
        nfev: 7
        njev: 1
[0.004954477179025791, 0.07688714601682278, 0.01745841189584378]


# Оценка ошибки методом Монте-Карло

Возмущаем `ecw` и `jdh` согласно `ecw_sigma` и `jdh_sigma`, строим
графики найденных решений и вычисляем матрицу ковариации.

In [44]:
mcrl_start = time.perf_counter()
Niter = 1000
ress = np.empty((3, Niter))
csts = np.empty(Niter)
for i in tqdm.trange(Niter):
    ecwr = np.random.normal(ecosw[:, a_ecw], ecosw[:, a_ecw_sigma])
    jdhr = np.random.normal(ecosw[:, a_jdh], ecosw[:, a_jdh_sigma])
    resr = solsq(x0, bounds, ecw_=ecwr, jdh_=jdhr, rcnt_=rcnt)
    ress[:, i] = resr.x
    csts[i] = resr.cost
mcrl_stop = time.perf_counter()

100%|█████████████████████████████████████████████| 1000/1000 [01:33<00:00, 10.70it/s]


In [45]:
cov = np.cov(ress)
err = np.sqrt(np.diag(cov))
with np.printoptions(precision=5, suppress=True):
    print(err)
with np.printoptions(precision=2):
    print(cov)
print(f"{data_set_id}, {method_id[1]}, N={ress.shape[1]}, {weight_txt}")
for i in range(3):
    print(f"{method_id[i]:4} = {res.x[i]: .5f} ± {err[i]:.5f} (1𝜎, 68%)")
    if "w0'" == method_id[i]:
        print(f"{'w0':4} = {res.x[i] + add2cos[i]: .5f} ± {err[i]:.5f} (1𝜎, 68%)")

[0.05183 3.40798 0.04177]
[[ 2.69e-03 -1.02e-02 -2.61e-05]
 [-1.02e-02  1.16e+01  7.21e-03]
 [-2.61e-05  7.21e-03  1.74e-03]]
ecosw 260327, w0', N=1000, Взвешенное
e    =  0.00495 ± 0.05183 (1𝜎, 68%)
w0'  =  0.07689 ± 3.40798 (1𝜎, 68%)
w0   = -1.49391 ± 3.40798 (1𝜎, 68%)
dw   =  0.01746 ± 0.04177 (1𝜎, 68%)


In [46]:
# wghs = 1./np.sqrt(csts)  # [0.08559 2.47188 0.03905]
#                          # avg - res.x  = [0.0204  0.0812  0.00121]
#                          # avg - res.x  = [ 0.00225  0.86033 -0.01126]
wghs = 1./csts             # [0.077   2.51628 0.03963]
                           # avg - res.x  = [0.01635 0.01307 0.0006 ]
                           # avg - res.x  = [ 0.00196  0.78847 -0.01385]
wcov = np.cov(ress, aweights=wghs)
werr = np.sqrt(np.diag(wcov))
with np.printoptions(precision=5, suppress=True):
    print(werr)
with np.printoptions(precision=2):
    print(wcov)
print(f"{data_set_id}, {method_id[1]}, N={ress.shape[1]}, {weight_txt}")
for i in range(3):
    print(f"{method_id[i]:4} = {res.x[i]: .5f} ± {werr[i]:.5f} (1𝜎, 68%)")
    if "w0'" == method_id[i]:
        print(f"{'w0':4} = {res.x[i] + add2cos[i]: .5f} ± {werr[i]:.5f} (1𝜎, 68%)")
# [0.08827 2.45221 0.03924]

[0.04178 3.36647 0.04069]
[[ 1.75e-03 -6.26e-03 -1.09e-05]
 [-6.26e-03  1.13e+01  3.79e-03]
 [-1.09e-05  3.79e-03  1.66e-03]]
ecosw 260327, w0', N=1000, Взвешенное
e    =  0.00495 ± 0.04178 (1𝜎, 68%)
w0'  =  0.07689 ± 3.36647 (1𝜎, 68%)
w0   = -1.49391 ± 3.36647 (1𝜎, 68%)
dw   =  0.01746 ± 0.04069 (1𝜎, 68%)


In [47]:
with np.printoptions(precision=5, suppress=True):
    print(wcov-cov)
    print(werr-err)

[[-0.00094  0.00396  0.00002]
 [ 0.00396 -0.28126 -0.00342]
 [ 0.00002 -0.00342 -0.00009]]
[-0.01005 -0.04152 -0.00108]


Среднее, по результатам Монте-Карло, отличается, но
в пределах ошибок. Примерно, так и должно быть, мы же возмущаем
входные данные.

Похоже взвешивание на `1./csts` немного лучше, но...

In [48]:
with np.printoptions(precision=5, suppress=True):
    avg = np.average(ress, axis=1)
    print(f"{'avg':12} = {avg}")
    print(f"{'avg - res.x':12} = {avg - res.x}")

avg          = [ 0.01297 -0.01786  0.00476]
avg - res.x  = [ 0.00802 -0.09475 -0.0127 ]


In [49]:
with np.printoptions(precision=5, suppress=True):
    avg = np.average(ress, axis=1, weights=wghs)
    print(f"{'avg':12} = {avg}")
    print(f"{'avg - res.x':12} = {avg - res.x}")

avg          = [ 0.01051 -0.18591  0.00345]
avg - res.x  = [ 0.00556 -0.26279 -0.01401]


In [50]:
cm = plt.colormaps.get_cmap('RdYlGn_r')
cm_scale = 4.

fig, ax = plt.subplots(4)
k = 0
for i in range(2):
    for j in range(i + 1, 3):
        ax[k].scatter(ress[i, :], ress[j, :], 2, cm(csts/cm_scale),
                   label=f"{method_id[i]}×{method_id[j]}")
        ax[k].legend()
        ax[k].set_xlabel(f"{method_id[i]}")
        ax[k].set_ylabel(f"{method_id[j]}")
        k += 1
n, bins, patches = ax[3].hist(csts, density=True, bins=50)
for bc, p in zip(0.5 * (bins[:-1] + bins[1:]), patches):
    plt.setp(p, 'facecolor', cm(bc/cm_scale))

fig.suptitle(f"{data_set_id}, {method_id[1]}, N={ress.shape[1]}, {weight_txt}"
             f"\n{time.strftime("%x %X", time.localtime())}"
             f" ({mcrl_stop - mcrl_start:.1f})")

Text(0.5, 0.98, "ecosw 260327, w0', N=1000, Взвешенное\n03.04.2026 05:43:41 (93.5)")

In [51]:
np.min(csts), np.max(csts)

(0.09949847969097957, 12.33527653445842)

# Резюме
Явно гарантированного ответа нет.  Вроде как, видно, что при
определённых сочетаниях $\omega_0$ ($\omega_0'$) и `dw`, данные 
могут быть объяснены при любом `e`.

Более вероятно:

In [52]:
print(f"{data_set_id}, {method_id[1]}, N={ress.shape[1]}, {weight_txt}")
for i in range(3):
    print(f"{method_id[i]:4} = {res.x[i]: .5f} ± {err[i]:.5f} (1𝜎, 68%)")
    if "w0'" == method_id[i]:
        print(f"{'w0':4} = {res.x[i] + add2cos[i]: .5f} ± {err[i]:.5f} (1𝜎, 68%)")

ecosw 260327, w0', N=1000, Взвешенное
e    =  0.00495 ± 0.05183 (1𝜎, 68%)
w0'  =  0.07689 ± 3.40798 (1𝜎, 68%)
w0   = -1.49391 ± 3.40798 (1𝜎, 68%)
dw   =  0.01746 ± 0.04177 (1𝜎, 68%)


Но может быть и:

In [81]:
x02 = x0.copy()
bounds2 = bounds.copy()
if "ecosw 260327" == data_set_id:
    x02[x_w0] = 1.5 - add2cos[x_w0]
    bounds2[0, x_w0] = 1.3 - add2cos[x_w0]
    bounds2[1, x_w0] = 1.7 - add2cos[x_w0]
    bounds2[0, x_dw] = -0.1
res2 = solsq(x02, bounds2, rcnt_=rcnt*5)
res2, bounds2

(     message: `ftol` termination condition is satisfied.
      success: True
       status: 2
          fun: [ 1.676e-01  2.409e-02 -5.680e-01 -1.623e-01  2.176e-02
                 3.790e-01 -3.683e-01]
            x: [ 4.991e-03  3.100e+00 -2.550e-04]
         cost: 0.32870510291837035
          jac: [[ 8.474e+00 -1.024e+00 -0.000e+00]
                [ 1.297e+02 -1.486e+00 -2.154e+03]
                ...
                [ 9.744e+02  1.680e+00  1.227e+04]
                [ 7.064e+02  2.009e+00  1.613e+04]]
         grad: [ 5.413e-09 -5.544e-09 -1.379e-05]
   optimality: 1.3793001016892958e-05
  active_mask: [0 0 0]
         nfev: 19
         njev: 12,
 array([[ 1.00000000e-07,  2.87079633e+00, -1.00000000e-01],
        [ 5.00000000e-01,  3.27079633e+00,  1.00000000e-01]]))

In [82]:
print(f"{data_set_id}, {method_id[1]}, N={ress.shape[1]}, {weight_txt}")
for i in range(3):
    print(f"{method_id[i]:4} = {res2.x[i]: .5f} ± {err[i]:.5f} (1𝜎, 68%)")
    if "w0'" == method_id[i]:
        print(f"{'w0':4} = {res2.x[i] + add2cos[i]: .5f} ± {err[i]:.5f} (1𝜎, 68%)")

ecosw 260327, w0', N=1000, Взвешенное
e    =  0.00499 ± 0.05183 (1𝜎, 68%)
w0'  =  3.10031 ± 3.40798 (1𝜎, 68%)
w0   =  1.52952 ± 3.40798 (1𝜎, 68%)
dw   = -0.00025 ± 0.04177 (1𝜎, 68%)
